# 🏁 Capstone — audit and document a model

**The brief.** A university wants a model that predicts which applicants will pass a digital-skills certification, to help decide who receives a funded study place. You are the analyst. Your job is to:

1. **Build** the best model you can *justify* — your choice of features and model
2. **Audit** it with the measurements from the ethics labs
3. **Document** it in a *model card* that a regulator, a journalist or an applicant could read
4. **Export** your DataLab portfolio — your model card plus every judgment you wrote in the ethics labs

There is no single right answer. What matters is that your decisions follow from your evidence, and that your documentation is honest about what the model gets wrong.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

%run datalab_ethics.py

df = pd.read_csv("data/student_digital_skills.csv")
df.head()

## Step 1 — Your design choices

Edit the settings below, then run the next two cells.

- **`FEATURES`** — which columns may the model use? Available: `age`, `hours_online_learning`, `digital_skill_score`, `prior_ict_courses`, `has_own_laptop`. Including or excluding `has_own_laptop` is a decision you will have to defend.
- **`MODEL_CHOICE`** — `"tree"`, `"logistic"` or `"forest"`.
- **`TREE_DEPTH`** — only used by the tree.

In [ ]:
FEATURES = ["age", "hours_online_learning", "digital_skill_score", "prior_ict_courses"]
MODEL_CHOICE = "tree"     # "tree", "logistic" or "forest"
TREE_DEPTH = 4

In [ ]:
ALLOWED = ["age", "hours_online_learning", "digital_skill_score",
           "prior_ict_courses", "has_own_laptop"]
unknown = [f for f in FEATURES if f not in ALLOWED]
if unknown or not FEATURES:
    raise ValueError(f"FEATURES must be a non-empty list chosen from {ALLOWED}; "
                     f"these are not available: {unknown}")

def build(choice):
    if choice == "tree":
        return DecisionTreeClassifier(max_depth=TREE_DEPTH, random_state=42)
    if choice == "logistic":
        return make_pipeline(StandardScaler(), LogisticRegression())
    if choice == "forest":
        return RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    raise ValueError(f"MODEL_CHOICE must be 'tree', 'logistic' or 'forest', not {choice!r}")

X, y = df[FEATURES], df["passed_certification"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = build(MODEL_CHOICE)
cv = cross_val_score(model, X_train, y_train, cv=5).mean()
model.fit(X_train, y_train)

baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Model: {MODEL_CHOICE}   Features: {FEATURES}")
print(f"Cross-validated accuracy: {cv:.0%}   Test accuracy: {model.score(X_test, y_test):.0%}   Always-'pass' baseline: {baseline:.0%}")

## Step 2 — Audit it

Run both measurements from the ethics labs. Audit by laptop ownership **whether or not** your model uses that column — auditors check the people affected, not just the inputs.

In [ ]:
laptop = df.loc[X_test.index, "has_own_laptop"].map({1: "has laptop", 0: "no laptop"})

group_performance(model, X_test, y_test, by=laptop)
print()
selection_rate_ratio(model.predict(X_test), groups=laptop)

## Step 3 — Iterate

Go back to **Step 1**, change one thing, and re-run Steps 1 and 2. Some experiments worth trying:

- Add `has_own_laptop` to `FEATURES`. Does the gap shrink, grow, or stay put?
- Try each `MODEL_CHOICE`. Which gives the smallest gap? At what cost in accuracy?
- Try a very shallow tree (`TREE_DEPTH = 2`).

Keep notes: a good model card mentions what you tried and why you rejected it.

## Step 4 — Write the model card

A **model card** is the standard short document that travels with a model: what it is for, how well it works, for whom it works badly. It is the practical format for what the EU AI Act asks providers of high-risk systems to document — intended purpose, accuracy and known limitations (Art. 11 and Annex IV; Art. 13).

The numbers fill themselves in from the model you just built. **The four written fields are yours.** A few honest sentences each.

**What a strong model card shows:**

- you know how much your model beats the baseline — and say so plainly
- the audit is reported as found, including what looks bad
- the limitations name **who** the model serves poorly, and why
- the recommendation follows from the evidence above it

In [ ]:
card = model_card(
    model, X_test, y_test, audit_by=laptop,
    model_name=f"{MODEL_CHOICE} (features: {len(FEATURES)})",
    features=FEATURES,
    intended_use="""Your answer: what is this model for, who would use it,
        and what decision does it inform?""",
    out_of_scope="""Your answer: what must it NOT be used for?""",
    limitations="""Your answer: what did your audit find? For whom is the
        model unreliable, and why?""",
    recommendation="""Your answer: deploy, deploy with conditions, or do not
        deploy — and why?""",
)

## Step 5 — Export your portfolio

Your portfolio collects this model card and every judgment you wrote in the DataLab's ethics labs. First check what it contains:

In [ ]:
show_portfolio()

Then build the file. Put your name in the quotes if you want it on the portfolio — it stays on your device; nothing is uploaded.

In [ ]:
export_portfolio(name="")

## 🎓 What happens next

The downloaded file is **self-reported learning evidence**. It opens in any browser, and your browser's *Print → Save as PDF* turns it into a PDF if you need one.

To have it count toward a **EURIDICE microcredential**, submit it through a EURIDICE partner institution — see *Earning a microcredential* on the course site.

⚠️ Your portfolio lives in this browser's storage. Clearing the site's data deletes it — **download it whenever you add something.**

Congratulations on finishing the DataLab.